# Loop 1 · Manual: the agent skips the lookup

The Acme Cloud support agent runs on **OpenClaw**. Its behavior is one file, `AGENTS.md`, which the
harness keeps in the MLflow Prompt Registry and renders into OpenClaw's workspace for every call.
The policies it should answer from sit next to it in `knowledge/`. The **MLflow connector**
(`connectors/openclaw-mlflow`) logs every OpenClaw run as its own trace, with an `LLM` span per
model call and a `TOOL` span per tool call.

Four loops, each finding one failure and fixing it with measurable evidence. v1 ships with the
first two; the other two arrive later, when other teams edit AGENTS.md and ship it without the
gate. Each loop hands more of the work to the harness:

| Loop | Failure | Signal | Store | Update | Promotes |
|---|---|---|---|---|---|
| 1 | Skips the lookup | you | you | you | you |
| 2 | Makes things up | you + a grounding judge | triage + approve | the analyst | you |
| 3 | Unsafe account help, from another team's edit | you + an aligned judge, must-pass check | triage + approve | an optimizer | you |
| 4 | Bad escalation, from another team's edit | a 50% label sample + scheduled scoring | `cycle` | `cycle` | the gate |

In this loop a person does every step. All results come from `session/store/` (the precomputed
run); `RUNBOOK.md` has the same steps as `selfimprove` commands.

In [ ]:
import _session as s

experiment_id = s.setup(1)
r = s.results(1)

## 1. Find: what v1 tells the agent

Look at the first bullet and the quick facts. Then compare them with `knowledge/`: refunds are
14 days, reset links last 15 minutes, lockouts 30 minutes, and Pro has 10 seats.

In [ ]:
from selfimprove import instructions

print(instructions.load(1).template)

## 2. Signal: production traffic, as the reviewers saw it

Eight customer questions went to production on v1. Each one is a trace. The reviewers (you, in
the room) labeled each answer pass or fail and attached the correct answer to every failure.

In [ ]:
before = s.loop_traces(1, "before")
s.show(s.answers(before))

Why did it fail? The harness trace only says "OpenClaw answered". The connector's trace for the
same run shows what OpenClaw actually did: one model call and **no `read`**. The harness links
the two with the run's session key (`openclaw.session_key` → `mlflow.trace.session`).

In [ ]:
s.tree(before[0])

That observation is cheap to turn into code. Loop 1's verifier (`session/criteria/loop1.yaml`) adds
two checks:

* `read_knowledge`: the connector's trace must have a successful `read` of a `knowledge/` file
* `no_stale_facts`: **must-pass**, no quoted stale fact (30-day, 24 hours, 25 seats, 1 hour)

`selfimprove score` ran them, and the `quality` judge, on every production trace:

In [ ]:
s.show(s.verdicts(before))

In [ ]:
s.show(s.failure_rate(before).to_frame("failure rate"), pct=True)

## 3. Store: the failures become tests

Each labeled failure becomes a dataset row: the question as input, the reviewer's correct answer
as the expectation, and its provenance as tags. A hash of the question puts about one in five in
the **holdout**, which only the gate ever sees. The eight golden holdout cases were there from
the start (`session/golden.yaml`).

In [ ]:
from selfimprove import store

print(r["dataset"])
df = store.dataset().to_df()
df = df[df["tags"].map(lambda t: t.get("source") == "golden" or t.get("author") == "you")]  # as of loop 1
s.show(df.assign(question=df["inputs"].map(lambda i: i["question"][:70]),
                 split=df["tags"].map(lambda t: t.get("split")),
                 source=df["tags"].map(lambda t: t.get("source")),
                 author=df["tags"].map(lambda t: t.get("author")))[["question", "split", "source", "author"]])

## 4. Update: a person writes v2

The fix names the four policy files (OpenClaw's `read` can't list a folder) and deletes the
stale quick facts. Nothing else changes: the other three flaws are next loops' work. It is
registered as a new version; `production` still points at v1.

In [ ]:
print(instructions.diff(r["production_before"], r["candidate"]))

## 5. Gate

Production and the candidate answer every dataset row fresh, scored by the same verifier. The
candidate ships only if every must-pass scorer is at 100%, it is not worse on train, it is not
worse on the holdout, and it is better somewhere.

In [ ]:
print(r["gate"]["outcome"], "-", r["gate"]["message"])
s.show(s.gate_table(1), pct=True)

## 6. Measure: the same traffic, after the promotion

A person promoted v2. The same eight questions went to production again and were reviewed again:

In [ ]:
s.show(s.before_after(1).astype({"failure_rate": float}), pct=False)

In [ ]:
after = s.loop_traces(1, "after")
s.show(s.answers(after))

In [ ]:
s.tree(after[0])

**What a person did in this loop:** everything. Read traces, labeled them, turned failures into
tests, wrote the fix, read the gate's table, promoted. The next loop hands the store and update
steps to the harness. Open the MLflow UI (`Traces`, `Prompts`, `Datasets`, and the `gate` run) to
see the same evidence there.